# Entropy- and Free-Energy–Driven Criticality Modeling for Quantum Wireless Communication Systems

This notebook presents a publication-ready proof of concept for a **Quantum Communication Criticality Index (QCCI)** designed to assess degradation in noisy quantum communication settings. The central idea is that communication instability can be characterized through a joint analysis of **quantum entropy**, **channel loss**, **decoherence**, and a lightweight **decision/control term**.

The notebook simulates noisy two-qubit communication states using Qiskit Aer, computes normalized **von Neumann entropy**, derives a nonlinear **capacity proxy**, and constructs the proposed **QCCI**. It then studies whether QCCI can detect failure regimes earlier than a simpler classical baseline risk indicator. In addition to the core results, the notebook generates figures, summary tables, failure-regime diagnostics, and a final aggregated text report stored on Google Drive.


This code cell mounts Google Drive, creates the required output directories, and ensures that all generated figures, tables, and text summaries are stored in the exact `Outputs` structure required for reproducibility.


In [ ]:
# ============================================================
# 1. GOOGLE DRIVE MOUNT + OUTPUT DIRECTORIES
# ============================================================

import warnings
from pathlib import Path
import sys
import subprocess
import importlib

warnings.filterwarnings("ignore")

IN_COLAB = False
try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    BASE_OUTPUT = Path('/content/drive/MyDrive/Outputs')
else:
    BASE_OUTPUT = Path.cwd() / 'Outputs'

FIG_DIR = BASE_OUTPUT / 'figures'
TABLE_DIR = BASE_OUTPUT / 'tables'

for p in [BASE_OUTPUT, FIG_DIR, TABLE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Google Colab detected:", IN_COLAB)
print("Base output directory:", BASE_OUTPUT)
print("Figures directory:", FIG_DIR)
print("Tables directory:", TABLE_DIR)


This code cell installs any missing Python dependencies needed for the notebook. The goal is to make the notebook runnable in a standard Google Colab environment with minimal manual intervention.


In [ ]:
# ============================================================
# 2. AUTO-INSTALL REQUIRED PACKAGES
# ============================================================

def pip_install(package_name):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

required_packages = [
    "numpy",
    "pandas",
    "matplotlib",
    "qiskit",
    "qiskit-aer",
]

for pkg in required_packages:
    try:
        importlib.import_module(pkg.replace("-", "_"))
        print(f"{pkg} already installed")
    except Exception:
        print(f"Installing {pkg} ...")
        pip_install(pkg)

importlib.invalidate_caches()
print("Dependency check completed.")


This code cell imports the scientific and quantum-computing libraries used throughout the notebook and sets the random seed for reproducibility.


In [ ]:
# ============================================================
# 3. IMPORTS + RANDOM SEED
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import DensityMatrix, entropy as vn_entropy, state_fidelity
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, phase_damping_error, amplitude_damping_error

SEED = 42
np.random.seed(SEED)

print("Imports loaded successfully.")


This code cell defines the QCCI weights and the thresholds used to categorize system states into **stable**, **risky**, and **failure**. These settings are intentionally tuned to make failure dynamics visible in the exploratory proof of concept.


In [ ]:
# ============================================================
# 4. THEORY CONFIGURATION
# ============================================================

ALPHA = 0.40
BETA = 0.40
GAMMA = 0.30
DELTA = 0.05

STABILITY_THRESHOLD = 0.55
FAILURE_THRESHOLD = 0.80

print("QCCI weights:")
print({"alpha": ALPHA, "beta": BETA, "gamma": GAMMA, "delta": DELTA})
print("Thresholds:")
print({"stability_threshold": STABILITY_THRESHOLD, "failure_threshold": FAILURE_THRESHOLD})


This code cell defines the reference quantum communication state, the noisy channel models, and the metrics used in the study. In particular, it implements:

- a simple entangled two-qubit reference state,
- noisy simulation with depolarizing, phase-damping, and amplitude-damping errors,
- a normalized entropy metric,
- a nonlinear capacity proxy,
- the proposed QCCI,
- state classification and early-warning calculations.


In [ ]:
# ============================================================
# 5. QUANTUM CIRCUIT AND METRIC HELPERS
# ============================================================

def build_reference_circuit():
    qc = QuantumCircuit(2)
    qc.h(0)
    qc.cx(0, 1)
    qc.save_density_matrix()
    return qc

def build_noise_model(depolarizing_lambda=0.0, phase_gamma=0.0, amplitude_gamma=0.0):
    noise_model = NoiseModel()

    if depolarizing_lambda > 0:
        err1 = depolarizing_error(min(depolarizing_lambda, 1.0), 1)
        err2 = depolarizing_error(min(depolarizing_lambda, 1.0), 2)
        noise_model.add_all_qubit_quantum_error(err1, ["h", "x", "sx", "rz"])
        noise_model.add_all_qubit_quantum_error(err2, ["cx"])

    if phase_gamma > 0:
        phase_err = phase_damping_error(min(phase_gamma, 1.0))
        noise_model.add_all_qubit_quantum_error(phase_err, ["h", "x", "sx", "rz"])

    if amplitude_gamma > 0:
        amp_err = amplitude_damping_error(min(amplitude_gamma, 1.0))
        noise_model.add_all_qubit_quantum_error(amp_err, ["h", "x", "sx", "rz"])

    return noise_model

def simulate_density_matrix(qc, noise_model=None, seed=SEED):
    backend = AerSimulator(method="density_matrix", noise_model=noise_model, seed_simulator=seed)
    tqc = transpile(qc, backend, seed_transpiler=seed)
    result = backend.run(tqc, shots=1).result()
    rho = result.data(0)["density_matrix"]
    return DensityMatrix(rho)

def channel_loss_metric(rho, reference_rho):
    fid = state_fidelity(reference_rho, rho)
    loss = 1.0 - float(np.clip(fid, 0.0, 1.0))
    return float(np.clip(loss, 0.0, 1.0))

def entropy_metric(rho):
    s = float(np.real(vn_entropy(rho, base=2)))
    return float(np.clip(s / 2.0, 0.0, 1.0))

def capacity_proxy_from_entropy(entropy_norm):
    return float(np.clip(np.exp(-3.0 * entropy_norm), 0.0, 1.0))

def qcci(channel_loss, entropy_norm, decoherence_level, decision_score):
    value = (
        ALPHA * channel_loss
        + BETA * entropy_norm
        + GAMMA * decoherence_level
        - DELTA * decision_score
    )
    return float(np.clip(value, 0.0, 1.0))

def classify_state(qcci_value):
    if qcci_value >= FAILURE_THRESHOLD:
        return "failure"
    if qcci_value >= STABILITY_THRESHOLD:
        return "risky"
    return "stable"

def early_warning_margin(qcci_value):
    return float(FAILURE_THRESHOLD - qcci_value)

print("Helper functions ready.")


This code cell generates the ideal reference state and reports its baseline entropy, capacity proxy, and channel loss. These reference values are used to compare the noisy cases later in the notebook.


In [ ]:
# ============================================================
# 6. BASELINE REFERENCE STATE
# ============================================================

reference_qc = build_reference_circuit()
reference_rho = simulate_density_matrix(reference_qc, noise_model=None)

reference_entropy = entropy_metric(reference_rho)
reference_capacity = capacity_proxy_from_entropy(reference_entropy)
reference_loss = channel_loss_metric(reference_rho, reference_rho)

print("Reference normalized entropy:", round(reference_entropy, 6))
print("Reference capacity proxy:", round(reference_capacity, 6))
print("Reference channel loss:", round(reference_loss, 6))


This code cell runs the main noise-sweep experiment. It varies depolarizing, phase-damping, and amplitude-damping noise, simulates the resulting density matrices, computes entropy and capacity, evaluates QCCI, and stores the outputs in a structured table.


In [ ]:
# ============================================================
# 7. FAILURE-DYNAMICS NOISE SWEEP
# ============================================================

rows = []

depolarizing_grid = np.linspace(0.0, 1.2, 25)

for dep_raw in depolarizing_grid:
    dep = float(min(dep_raw, 1.0))
    for phase in [0.0, 0.3, 0.6, 0.9]:
        for amp in [0.0, 0.3, 0.6]:
            noise_model = build_noise_model(
                depolarizing_lambda=dep,
                phase_gamma=float(phase),
                amplitude_gamma=float(amp),
            )
            rho = simulate_density_matrix(reference_qc, noise_model=noise_model)

            ch_loss = channel_loss_metric(rho, reference_rho)
            ent = entropy_metric(rho)
            cap = capacity_proxy_from_entropy(ent)

            decoherence_level = float(np.clip(dep_raw + phase + amp, 0.0, 1.5))
            decoherence_norm = decoherence_level / 1.5

            decision_score = float(np.clip(1.0 - 1.2 * decoherence_norm, 0.0, 1.0))

            qcci_value = qcci(
                channel_loss=ch_loss,
                entropy_norm=ent,
                decoherence_level=decoherence_norm,
                decision_score=decision_score,
            )

            classical_baseline_risk = float(np.clip(0.5 * dep + 0.25 * phase + 0.25 * amp, 0.0, 1.0))

            rows.append({
                "depolarizing_raw": float(dep_raw),
                "depolarizing_used": float(dep),
                "phase_damping": float(phase),
                "amplitude_damping": float(amp),
                "decoherence_level_raw": decoherence_level,
                "decoherence_level_norm": decoherence_norm,
                "decision_score": decision_score,
                "channel_loss": ch_loss,
                "entropy_norm": ent,
                "capacity_proxy": cap,
                "classical_baseline_risk": classical_baseline_risk,
                "QCCI": qcci_value,
                "early_warning_margin": early_warning_margin(qcci_value),
                "state": classify_state(qcci_value),
            })

results = pd.DataFrame(rows)

print("Noise sweep completed.")
print("Shape:", results.shape)
results.head()


This code cell saves the main experiment table and creates a state-level summary table. These tables give a compact overview of how the stable, risky, and failure regimes differ.


In [ ]:
# ============================================================
# 8. MAIN TABLES
# ============================================================

results_path = TABLE_DIR / "qcci_noise_sweep_results.csv"
summary_path = TABLE_DIR / "qcci_state_summary.csv"

results.to_csv(results_path, index=False)

summary = (
    results.groupby("state")
    .agg(
        count=("state", "size"),
        avg_qcci=("QCCI", "mean"),
        avg_entropy=("entropy_norm", "mean"),
        avg_capacity=("capacity_proxy", "mean"),
        avg_loss=("channel_loss", "mean"),
    )
    .reset_index()
)

summary.to_csv(summary_path, index=False)

print("Saved:")
print("-", results_path)
print("-", summary_path)
summary


This code cell reports state counts and correlations among the main variables. These descriptive statistics help verify whether the constructed QCCI behaves consistently with the underlying entropy and capacity trends.


In [ ]:
# ============================================================
# 9. CORE DESCRIPTIVE STATISTICS
# ============================================================

print("State counts:")
print(results["state"].value_counts())

corr_cols = [
    "depolarizing_raw", "phase_damping", "amplitude_damping",
    "channel_loss", "entropy_norm", "capacity_proxy", "QCCI"
]

print("\nCorrelations:")
print(results[corr_cols].corr().round(3))


This code cell plots how normalized entropy changes as depolarizing noise increases in the cleanest slice of the experiment. The figure is intended to show the growth of uncertainty under stronger channel degradation.


In [ ]:
# ============================================================
# 10. FIGURE 1 - ENTROPY VS DEPOLARIZING NOISE
# ============================================================

fig1_path = FIG_DIR / "fig1_entropy_vs_depolarizing.png"

plot_df = results[
    (results["phase_damping"] == 0.0) &
    (results["amplitude_damping"] == 0.0)
].sort_values("depolarizing_raw")

plt.figure(figsize=(8, 5))
plt.plot(plot_df["depolarizing_raw"], plot_df["entropy_norm"], marker="o")
plt.xlabel("Depolarizing Noise Level")
plt.ylabel("Normalized von Neumann Entropy")
plt.title("Entropy Growth Under Depolarizing Noise")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(fig1_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved figure:", fig1_path)


This code cell plots the nonlinear capacity proxy as noise increases. It is designed to make communication collapse more visually evident than a purely linear proxy.


In [ ]:
# ============================================================
# 11. FIGURE 2 - CAPACITY PROXY VS DEPOLARIZING NOISE
# ============================================================

fig2_path = FIG_DIR / "fig2_capacity_vs_depolarizing.png"

plt.figure(figsize=(8, 5))
plt.plot(plot_df["depolarizing_raw"], plot_df["capacity_proxy"], marker="o")
plt.xlabel("Depolarizing Noise Level")
plt.ylabel("Capacity Proxy")
plt.title("Nonlinear Capacity Collapse Under Depolarizing Noise")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(fig2_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved figure:", fig2_path)


This code cell plots QCCI against depolarizing noise and overlays the risk and failure thresholds. It provides a direct visual representation of when the system transitions from stable operation into risky and failure regimes.


In [ ]:
# ============================================================
# 12. FIGURE 3 - QCCI VS DEPOLARIZING NOISE
# ============================================================

fig3_path = FIG_DIR / "fig3_qcci_vs_depolarizing.png"

plt.figure(figsize=(8, 5))
plt.plot(plot_df["depolarizing_raw"], plot_df["QCCI"], marker="o", label="QCCI")
plt.axhline(STABILITY_THRESHOLD, linestyle="--", label="Risk Threshold")
plt.axhline(FAILURE_THRESHOLD, linestyle=":", label="Failure Threshold")
plt.xlabel("Depolarizing Noise Level")
plt.ylabel("QCCI")
plt.title("QCCI Escalation Under Depolarizing Noise")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(fig3_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved figure:", fig3_path)


This code cell compares QCCI with a simpler classical baseline risk measure under a mixed-noise regime. The purpose is to test whether QCCI reaches the failure region earlier.


In [ ]:
# ============================================================
# 13. FIGURE 4 - QCCI VS CLASSICAL BASELINE UNDER MIXED NOISE
# ============================================================

fig4_path = FIG_DIR / "fig4_qcci_vs_classical_baseline.png"

compare_df = results[
    (results["phase_damping"] == 0.6) &
    (results["amplitude_damping"] == 0.6)
].sort_values("depolarizing_raw")

plt.figure(figsize=(8, 5))
plt.plot(compare_df["depolarizing_raw"], compare_df["QCCI"], marker="o", label="QCCI")
plt.plot(compare_df["depolarizing_raw"], compare_df["classical_baseline_risk"], marker="s", label="Classical baseline risk")
plt.axhline(FAILURE_THRESHOLD, linestyle=":", label="Failure Threshold")
plt.xlabel("Depolarizing Noise Level")
plt.ylabel("Risk / Criticality")
plt.title("QCCI vs Classical Baseline Under Mixed Noise")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(fig4_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved figure:", fig4_path)


This code cell adds the direct QCCI-versus-capacity visualization requested for stronger reviewer-facing evidence. It helps show that QCCI increases as communication capacity collapses.


In [ ]:
# ============================================================
# 14. FIGURE 5 - QCCI VS COMMUNICATION CAPACITY
# ============================================================

fig5_path = FIG_DIR / "fig5_qcci_vs_capacity.png"

plt.figure(figsize=(8, 5))
plt.scatter(results["capacity_proxy"], results["QCCI"], alpha=0.7)
plt.xlabel("Capacity Proxy")
plt.ylabel("QCCI")
plt.title("QCCI vs Communication Capacity")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(fig5_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved figure:", fig5_path)


This code cell determines the first depolarizing noise level at which QCCI and the classical baseline cross the failure threshold. This provides the quantitative early-warning comparison used in the conclusions.


In [ ]:
# ============================================================
# 15. EARLY-WARNING ANALYSIS
# ============================================================

qcci_cross = compare_df.loc[compare_df["QCCI"] >= FAILURE_THRESHOLD, "depolarizing_raw"]
base_cross = compare_df.loc[compare_df["classical_baseline_risk"] >= FAILURE_THRESHOLD, "depolarizing_raw"]

qcci_first = None if qcci_cross.empty else float(qcci_cross.iloc[0])
base_first = None if base_cross.empty else float(base_cross.iloc[0])

early_warning_df = pd.DataFrame({
    "metric": ["QCCI", "Classical baseline"],
    "first_failure_crossing_depolarizing_noise": [qcci_first, base_first],
})

early_warning_path = TABLE_DIR / "qcci_early_warning_comparison.csv"
early_warning_df.to_csv(early_warning_path, index=False)

print(early_warning_df)
print("\nSaved:", early_warning_path)


This code cell evaluates a few alternative QCCI weight settings. The goal is to test whether the overall conclusions remain stable when the balance between loss, entropy, and decoherence is adjusted.


In [ ]:
# ============================================================
# 16. SENSITIVITY ANALYSIS
# ============================================================

weight_sets = {
    "balanced": (0.40, 0.40, 0.30, 0.05),
    "entropy_focused": (0.30, 0.50, 0.30, 0.05),
    "loss_focused": (0.50, 0.30, 0.30, 0.05),
}

s_rows = []

for name, (a, b, g, d) in weight_sets.items():
    temp = results.copy()
    temp["QCCI_variant"] = np.clip(
        a * temp["channel_loss"]
        + b * temp["entropy_norm"]
        + g * temp["decoherence_level_norm"]
        - d * temp["decision_score"],
        0.0,
        1.0,
    )
    s_rows.append({
        "configuration": name,
        "mean_qcci": temp["QCCI_variant"].mean(),
        "max_qcci": temp["QCCI_variant"].max(),
        "fraction_failure": (temp["QCCI_variant"] >= FAILURE_THRESHOLD).mean(),
        "fraction_risky_or_worse": (temp["QCCI_variant"] >= STABILITY_THRESHOLD).mean(),
    })

sensitivity_df = pd.DataFrame(s_rows)
sensitivity_path = TABLE_DIR / "qcci_sensitivity_analysis.csv"
sensitivity_df.to_csv(sensitivity_path, index=False)

print("Saved:", sensitivity_path)
sensitivity_df


This code cell produces a pivoted mixed-noise table showing how QCCI varies jointly with depolarizing and phase-damping noise at a fixed amplitude-damping level.


In [ ]:
# ============================================================
# 17. PIVOT TABLE FOR MIXED NOISE
# ============================================================

fixed_amp = 0.6
heat_df = results[results["amplitude_damping"] == fixed_amp].copy()

pivot_qcci = heat_df.pivot_table(
    index="phase_damping",
    columns="depolarizing_raw",
    values="QCCI",
    aggfunc="mean"
).round(3)

pivot_path = TABLE_DIR / "qcci_pivot_mixed_noise.csv"
pivot_qcci.to_csv(pivot_path)

print("Saved:", pivot_path)
pivot_qcci


This code cell extracts all the failure-regime cases so that the severe degradation region can be examined separately if needed.


In [ ]:
# ============================================================
# 18. FAILURE-REGIME TABLE
# ============================================================

failure_cases = results[results["state"] == "failure"].copy().sort_values(
    ["QCCI", "depolarizing_raw"], ascending=[False, True]
)

failure_table_path = TABLE_DIR / "qcci_failure_cases.csv"
failure_cases.to_csv(failure_table_path, index=False)

print("Saved:", failure_table_path)
print("Number of failure cases:", len(failure_cases))
failure_cases.head(10)


This code cell adds the requested reviewer-oriented comparison table between **stable** and **failure** regimes. It summarizes the average entropy, capacity, QCCI, channel loss, and decoherence levels for those two extremes.


In [ ]:
# ============================================================
# 19. STABLE VS FAILURE COMPARISON TABLE
# ============================================================

stable_failure_table = (
    results[results["state"].isin(["stable", "failure"])]
    .groupby("state")
    .agg(
        entropy_norm=("entropy_norm", "mean"),
        capacity_proxy=("capacity_proxy", "mean"),
        QCCI=("QCCI", "mean"),
        channel_loss=("channel_loss", "mean"),
        decoherence_level_norm=("decoherence_level_norm", "mean"),
    )
    .T
)

stable_failure_table = stable_failure_table.rename(
    columns={"stable": "Stable", "failure": "Failure"}
).round(4)

stable_failure_path = TABLE_DIR / "qcci_stable_vs_failure_comparison.csv"
stable_failure_table.to_csv(stable_failure_path)

print("Saved:", stable_failure_path)
stable_failure_table


This code cell writes the aggregated textual report to the exact file path requested by you: `outputs_summary.txt` with an underscore, not a space. The report consolidates the main numerical findings from the notebook.


In [ ]:
# ============================================================
# 20. OUTPUTS SUMMARY TXT FILE
# Exact requested path:
# /content/drive/MyDrive/Outputs/outputs_summary.txt
# ============================================================

stable_pct = 100 * (results["state"] == "stable").mean()
risky_pct = 100 * (results["state"] == "risky").mean()
failure_pct = 100 * (results["state"] == "failure").mean()

corr_entropy_qcci = results["entropy_norm"].corr(results["QCCI"])
corr_capacity_qcci = results["capacity_proxy"].corr(results["QCCI"])

summary_text = f"""
QCCI PROOF-OF-CONCEPT SUMMARY
=============================

Total experiments: {len(results)}

State distribution:
- Stable:  {stable_pct:.2f}%
- Risky:   {risky_pct:.2f}%
- Failure: {failure_pct:.2f}%

Correlation analysis:
- Corr(entropy_norm, QCCI):   {corr_entropy_qcci:.4f}
- Corr(capacity_proxy, QCCI): {corr_capacity_qcci:.4f}

Early-warning comparison:
- QCCI first failure crossing (mixed-noise slice): {qcci_first}
- Classical baseline first failure crossing:        {base_first}

Interpretation:
- Higher noise increases entropy and channel loss.
- Higher entropy reduces the capacity proxy through a nonlinear collapse model.
- QCCI captures combined degradation from loss, entropy, and decoherence.
- Failure dynamics are explicitly induced to test critical-transition behavior.
- QCCI rises as communication capacity collapses.
- If QCCI crosses the failure threshold earlier than the classical baseline,
  it acts as a stronger early-warning metric in this proof of concept.
""".strip()

summary_file = BASE_OUTPUT / "outputs_summary.txt"
summary_file.write_text(summary_text, encoding="utf-8")

print(summary_text)
print("\nSaved summary:", summary_file)


This code cell demonstrates a simple action recommendation layer based on QCCI. While lightweight, it illustrates how the criticality index can support practical decision making.


In [ ]:
# ============================================================
# 21. OPTIONAL LIGHTWEIGHT DECISION LAYER
# ============================================================

def recommend_action(qcci_value):
    if qcci_value >= FAILURE_THRESHOLD:
        return "reroute_or_suspend"
    if qcci_value >= STABILITY_THRESHOLD:
        return "apply_error_mitigation"
    return "continue_normal_operation"

action_demo = compare_df[["depolarizing_raw", "phase_damping", "amplitude_damping", "QCCI"]].copy()
action_demo["recommended_action"] = action_demo["QCCI"].apply(recommend_action)

action_path = TABLE_DIR / "qcci_action_recommendations.csv"
action_demo.to_csv(action_path, index=False)

print("Saved:", action_path)
action_demo.head(10)


This final code cell lists the figures, tables, and text artifacts created by the notebook, making it easy to verify that all expected outputs were generated.


In [ ]:
# ============================================================
# 22. FINAL CHECK OF GENERATED ARTIFACTS
# ============================================================

print("Figures:")
for f in sorted(FIG_DIR.glob("*.png")):
    print("-", f.name)

print("\nTables:")
for f in sorted(TABLE_DIR.glob("*.csv")):
    print("-", f.name)

print("\nOther:")
for f in sorted(BASE_OUTPUT.glob("*.txt")):
    print("-", f.name)
